# DKD + TGSC: resnet32x4 -> resnet8x4

Attach [aj903501/tgsc-resnet32x4-try1](https://www.kaggle.com/datasets/aj903501/tgsc-resnet32x4-try1) with **Add Input** before running. Enable GPU and Internet. No GitHub token is required. Training uses only the supplied crop bank. Evaluation downloads only the verified 28 MB test split. The notebook checks the teacher/cache, trains for 240 epochs, evaluates every five epochs, and saves logs/plots. Use the same Dataset version and preparation environment across methods.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Edit this cell. Paths below are examples; use your attached artifact locations.
REPO_URL = "https://github.com/ahmad903501/tgsc-mdistiller.git"
PRIVATE_REPO = False
REVISION = 'b8cd039'
REPO_DIR = Path("/kaggle/working/mdistiller")
CONFIG_NAME = 'dkd_resnet32x4_to_resnet8x4.json'
GPU = 0
SEED = 27
EVAL_EVERY = 5  # Also evaluate at the final epoch or a requested early stop.
TEST_DATA = None  # Optional attached cifar100-test.npz; otherwise download only 28 MB.
CIFAR_SOURCE = 'download'
DATA_ROOT = Path("/kaggle/working/data")
TEACHER_SOURCE = 'kaggle'
TEACHER_DOWNLOAD_ROOT = Path("/kaggle/working/teacher_weights")
TEACHER_CHECKPOINT = Path("/kaggle/input/datasets/aj903501/tgsc-resnet32x4-try1/teacher_weights/cifar_teachers/resnet32x4_vanilla/ckpt_epoch_240.pth")
CACHE_ROOT = Path("/kaggle/input/datasets/aj903501/tgsc-resnet32x4-try1/tgsc_cache")
CACHE_WRITE_ROOT = Path("/kaggle/working/tgsc_cache")
OUTPUT_DIR = Path("/kaggle/working/tgsc_runs/dkd_resnet32x4_to_resnet8x4_seed27")
RESUME = None

TEACHER_NAME = 'resnet32x4'
DATASET_SLUG = 'tgsc-resnet32x4-try1'
DATASET_URL = 'https://www.kaggle.com/datasets/aj903501/tgsc-resnet32x4-try1'
INPUT_ROOT = Path("/kaggle/input/datasets/aj903501/tgsc-resnet32x4-try1")


In [ ]:
import json
from importlib.metadata import version

def resolve_inputs(roots):
    matches = set()
    for root in roots:
        if not root.is_dir():
            continue
        checkpoints = [p for p in root.rglob("ckpt_epoch_240.pth")
                       if p.parent.name == TEACHER_NAME + "_vanilla"]
        banks = []
        for path in root.rglob("manifest.json"):
            identity = json.loads(path.read_text())
            if identity.get("schema") == 2 and identity.get("teacher") == TEACHER_NAME:
                if all((path.parent / f"res{r}.pt").is_file() for r in (16,20,24,28,32)):
                    banks.append(path)
        for checkpoint in checkpoints:
            for bank in banks:
                matches.add((checkpoint.resolve(), bank.resolve()))
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one complete {TEACHER_NAME} teacher/bank pair; found {len(matches)}. "
            f"Attach {DATASET_URL} using Add Input, then rerun. "
            "The Dataset must contain extracted teacher_weights/ and tgsc_cache/ folders. "
            "If multiple versions/banks are attached, keep only the intended version.")
    return next(iter(matches))

TEACHER_CHECKPOINT, BANK_MANIFEST = resolve_inputs([
    INPUT_ROOT,
    Path("/kaggle/input") / DATASET_SLUG,
])
CACHE_ROOT = BANK_MANIFEST.parent.parent
identity = json.loads(BANK_MANIFEST.read_text())
for package_name, field in [("torch", "torch"), ("torchvision", "torchvision")]:
    installed = version(package_name)
    if installed != identity[field]:
        raise RuntimeError(
            f"{package_name}={installed}, but this cache was prepared with {identity[field]}. "
            "Use the preparation environment; cache identity checks must not be bypassed.")
print("Dataset:", DATASET_URL)
print("Teacher checkpoint:", TEACHER_CHECKPOINT)
print("Crop bank:", BANK_MANIFEST.parent)
print("Runtime revision:", REVISION)


In [ ]:
if not REPO_DIR.exists():
    if "YOUR_ACCOUNT" in REPO_URL:
        raise ValueError("Set REPO_URL to your pushed repository")
    def clone_repository():
        if not PRIVATE_REPO:
            subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
            return
        import os
        import tempfile
        from kaggle_secrets import UserSecretsClient
        # Keep the token out of URLs, notebook output, and files.
        clone_env = dict(os.environ, TGSC_GITHUB_TOKEN=UserSecretsClient().get_secret("GITHUB_TOKEN"),
                         GIT_TERMINAL_PROMPT="0")
        with tempfile.TemporaryDirectory() as auth_directory:
            askpass = Path(auth_directory) / "askpass.py"
            askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['TGSC_GITHUB_TOKEN'])\n")
            askpass.chmod(0o700)
            clone_env["GIT_ASKPASS"] = str(askpass)
            subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], env=clone_env, check=True)
    clone_repository()
    subprocess.run(["git", "checkout", REVISION], cwd=REPO_DIR, check=True)
else:
    wanted = subprocess.check_output(["git", "rev-parse", REVISION], cwd=REPO_DIR, text=True).strip()
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
    if wanted != actual:
        raise RuntimeError("Existing checkout differs from REVISION; select the intended checkout before running")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-tgsc.txt")], check=True)
print("Code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())


## Teacher, crop bank and test-only evaluation data

Training and preflight use the prepared crop bank and teacher checkpoint. Original CIFAR training files are not loaded or downloaded. `CIFAR_SOURCE="download"` allows only the verified 28 MB test-split download when training/evaluation needs it; `check` downloads no data. For an attached `cifar100-test.npz`, set `TEST_DATA` to its file path and `CIFAR_SOURCE="kaggle"`.

The teacher source can be `"kaggle"` (attached weights) or `"download"` (official combined teacher archive, selected checkpoint extracted). Keep existing TGSC crop/augmentation settings. Evaluation runs every `EVAL_EVERY=5` epochs, plus the final epoch or a requested early stop. Logs/checkpoints are saved every epoch; plots update on evaluation epochs.


In [ ]:
if TEACHER_SOURCE not in {"download", "kaggle"}:
    raise ValueError('TEACHER_SOURCE must be "download" or "kaggle"')
if CIFAR_SOURCE not in {"download", "kaggle"}:
    raise ValueError('CIFAR_SOURCE must be "download" or "kaggle"')
DOWNLOAD = CIFAR_SOURCE == "download"
if DOWNLOAD and DATA_ROOT.as_posix().startswith("/kaggle/input/"):
    raise ValueError("CIFAR download needs a writable DATA_ROOT, e.g. /kaggle/working/data")
if not DOWNLOAD and not (Path(TEST_DATA) if TEST_DATA is not None else DATA_ROOT / "cifar100-test.npz").is_file():
    raise FileNotFoundError("Attach cifar100-test.npz via TEST_DATA, or use CIFAR_SOURCE=download for the test-only download")
if TEACHER_SOURCE == "download":
    subprocess.run([
        sys.executable, str(REPO_DIR / "tools" / "download_teachers.py"),
        "--config", str(REPO_DIR / "configs" / "tgsc" / "cifar100" / CONFIG_NAME),
        "--output-dir", str(TEACHER_DOWNLOAD_ROOT),
    ], cwd=REPO_DIR, check=True)
    import json
    manifest = json.loads((TEACHER_DOWNLOAD_ROOT / "teacher_manifest.json").read_text())
    TEACHER_CHECKPOINT = TEACHER_DOWNLOAD_ROOT / manifest["relative_checkpoint"]
if not TEACHER_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Teacher checkpoint does not exist: {TEACHER_CHECKPOINT}")
print("Teacher checkpoint:", TEACHER_CHECKPOINT)
print("CIFAR-100:", CIFAR_SOURCE, DATA_ROOT)


In [ ]:
CONFIG = REPO_DIR / "configs" / "tgsc" / "cifar100" / CONFIG_NAME
RUNNER = REPO_DIR / "tools" / "run_tgsc.py"

def run_tgsc(command, *extra):
    args = [sys.executable, "-u", str(RUNNER), command,
            "--config", str(CONFIG), "--gpu", str(GPU), "--seed", str(SEED),
            "--data-root", str(DATA_ROOT), "--cache-root", str(CACHE_ROOT)]
    if command != "evaluate":
        args += ["--teacher-checkpoint", str(TEACHER_CHECKPOINT)]
    if command == "train":
        args += ["--eval-every", str(EVAL_EVERY)]
    if command in {"train", "evaluate"} and TEST_DATA is not None:
        args += ["--test-data", str(TEST_DATA)]
    if DOWNLOAD:
        args += ["--download"]
    args += [str(value) for value in extra]
    log_dir = CACHE_WRITE_ROOT if command == "prepare" else OUTPUT_DIR
    log_dir.mkdir(parents=True, exist_ok=True)
    with (log_dir / (command + ".log")).open("a", encoding="utf-8") as log:
        process = subprocess.Popen(args, cwd=REPO_DIR, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, args)
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise


## Check before training

Quick preflight samples 32 crop records per size and checks complete metadata, teacher/provenance compatibility and consistency with full-image cached labels. It requires no original training files and downloads no test data. Model loss/backward checks cover all crop sizes. Training fully validates each active stage. For exhaustive pixel preflight use `run_tgsc("check", "--full-cache-check")`.


In [ ]:
run_tgsc("check")


## Train or resume

The runner saves each completed epoch. Training uses the existing TGSC curriculum and default augmentation. No new baseline training is performed.


In [ ]:
extra = ["--output-dir", OUTPUT_DIR]
if RESUME is not None:
    extra += ["--resume", RESUME]
run_tgsc("train", *extra)


## Evaluate the exported model

SimKD exports its projector and reused teacher head as well as the backbone. Evaluation does not require teacher-backbone weights. Save notebook outputs to preserve checkpoints/logs for a later session.


In [ ]:
run_tgsc("evaluate", "--export", OUTPUT_DIR / "inference_best.pt")


## Saved logs and curves
Training metrics are logged every epoch; test metrics are measured every five epochs plus final/requested stop. Save the output directory: metrics.jsonl, metrics.csv, training_curves.png/.svg, run.json and model/checkpoint files. Missing test measurements are left blank.


In [ ]:
from IPython.display import Image, display
curve = OUTPUT_DIR / "training_curves.png"
if curve.is_file():
    display(Image(filename=str(curve)))
print("Saved outputs:", OUTPUT_DIR)
